### New inference: a model that transfers to Dallas

The first model (`airbnb_v1`) has a big gap: on the Austin + Fort Worth test set it beats the baseline by 29%, but on Dallas only by 13%.

**Why.** Three of its features mean something different in a new city (assignment 3.7: *"a feature that means something different in city C"*):

| Feature | Problem in Dallas |
|---|---|
| `latitude`, `longitude` | Dallas is outside the map the model learned. Trees cannot extrapolate, so Dallas gets the prices of the nearest edge of the training map (east Fort Worth). |
| `neighbourhood_cleansed` | Every Dallas neighbourhood is new, so this feature gives no information. |
| `city` | `dallas` is a value the model never saw. |

**What we change (model changes, not pipeline fixes):**

| # | Change | Why |
|---|---|---|
| 1 | Remove `latitude`, `longitude` | absolute coordinates do not transfer to a new city |
| 2 | Add `dist_center_km`: distance to the city's downtown | "close to downtown" means the same in every city |
| 3 | Remove `neighbourhood_cleansed` and `city` | the values in Dallas are all new |
| 4 | Add `n_amenities`, `has_pool`, `has_hot_tub` | the biggest Dallas errors are expensive listings; these features describe them |

**A fair test.** We choose the features **without Dallas**: we train on one training city and test on the other one (Austin → Fort Worth and Fort Worth → Austin), using only the train part. The test set and Dallas are used only once, at the end.

#### Configuration

In [0]:
import json

import pyspark.sql.functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
from pyspark.ml.regression import GBTRegressor

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
GOLD_FEATURES_TABLE = f"{CATALOG}.default.gold_listings_features"
V1_METRICS_TABLE = f"{CATALOG}.default.gold_model_metrics"
NEW_PREDICTIONS_TABLE = f"{CATALOG}.default.gold_price_predictions_v2"
NEW_METRICS_TABLE = f"{CATALOG}.default.gold_model_metrics_v2"

MODELS_VOLUME = "/Volumes/workspace/default/models"
V1_MODEL_DIR = f"{MODELS_VOLUME}/price_model_airbnb_v1"
V1_PREPROCESSING_JSON = f"{MODELS_VOLUME}/price_model_airbnb_v1_preprocessing.json"
V2_MODEL_DIR = f"{MODELS_VOLUME}/price_model_airbnb_v2"

TRAIN_CITIES = ["austin", "fort_worth"]
NEW_CITY = "dallas"
RANDOM_SEED = 67
MODEL_VERSION = "airbnb_v2"

# Downtown of each city (latitude, longitude). A new city needs one new line here.
CITY_CENTERS = {
    "austin": (30.2672, -97.7431),
    "fort_worth": (32.7555, -97.3308),
    "dallas": (32.7767, -96.7970),
}

#### Features

The old feature lists are the same as in `Texas_price_prediction`. The new lists remove the features that do not transfer and add the new ones.

In [0]:
V1_NUMERIC = [
    "latitude", "longitude", "accommodates", "bathrooms", "bedrooms", "beds",
    "calculated_host_listings_count", "minimum_nights", "maximum_nights",
    "availability_30", "availability_365", "estimated_occupancy_l365d",
    "number_of_reviews", "review_scores_rating", "review_scores_cleanliness",
    "review_scores_location", "reviews_per_month",
    "host_is_superhost", "bathroom_shared", "is_studio", "has_reviews",
]
V1_CATEGORICAL = ["city", "neighbourhood_cleansed", "property_type", "room_type"]

V2_NUMERIC = (
    [c for c in V1_NUMERIC if c not in ("latitude", "longitude")]
    + ["dist_center_km", "n_amenities", "has_pool", "has_hot_tub"]
)
V2_CATEGORICAL = ["property_type", "room_type"]

FEATURE_SETS = {
    "v1: old features": (V1_NUMERIC, V1_CATEGORICAL),
    "v2: transferable features": (V2_NUMERIC, V2_CATEGORICAL),
}

centers = spark.createDataFrame(
    [(city, lat, lon) for city, (lat, lon) in CITY_CENTERS.items()],
    "city string, center_lat double, center_lon double",
)


def add_features(df):
    amenities_text = F.lower(F.concat_ws(" ", "amenities"))
    df = (
        df.join(centers, on="city", how="left")
        # distance to downtown in km (haversine formula)
        .withColumn("dist_center_km", 6371 * 2 * F.asin(F.sqrt(
            F.pow(F.sin(F.radians(F.col("latitude") - F.col("center_lat")) / 2), 2)
            + F.cos(F.radians("latitude")) * F.cos(F.radians("center_lat"))
            * F.pow(F.sin(F.radians(F.col("longitude") - F.col("center_lon")) / 2), 2)
        )))
        .withColumn("n_amenities", F.when(F.col("amenities").isNull(), 0).otherwise(F.size("amenities")))
        .withColumn("has_pool", amenities_text.contains("pool"))
        .withColumn("has_hot_tub", amenities_text.contains("hot tub"))
    )
    for c in set(V1_NUMERIC + V2_NUMERIC):
        df = df.withColumn(c, F.col(c).cast("double"))
    return df


data = add_features(
    spark.table(GOLD_FEATURES_TABLE).filter(F.col("price") > 0)
).cache()

missing_centers = [r["city"] for r in data.filter(F.col("center_lat").isNull()).select("city").distinct().collect()]
assert not missing_centers, f"Add these cities to CITY_CENTERS: {missing_centers}"

display(
    data.groupBy("city")
    .agg(
        F.count("*").alias("listings"),
        F.round(F.expr("percentile_approx(dist_center_km, 0.5)"), 1).alias("median_km_to_downtown"),
        F.round(F.avg("has_pool"), 2).alias("share_with_pool"),
    )
    .orderBy("city")
)

city,listings,median_km_to_downtown,share_with_pool
austin,9759,5.1,0.4
dallas,5913,4.8,0.39
fort_worth,2151,6.4,0.18


#### Train / test split

The same split as in `Texas_price_prediction`: 80 / 20 per city, seed 67. Dallas is kept apart.

In [0]:
train_parts, test_parts = [], []
for city in TRAIN_CITIES:
    city_train, city_test = data.filter(F.col("city") == city).randomSplit([0.8, 0.2], seed=RANDOM_SEED)
    train_parts.append(city_train.withColumn("split", F.lit("train")))
    test_parts.append(city_test.withColumn("split", F.lit("test")))

train_df, test_df = train_parts[0], test_parts[0]
for part in train_parts[1:]:
    train_df = train_df.unionByName(part)
for part in test_parts[1:]:
    test_df = test_df.unionByName(part)

dallas_df = data.filter(F.col("city") == NEW_CITY).withColumn("split", F.lit("inference"))

print(f"Train: {train_df.count():,}   Test: {test_df.count():,}   Dallas: {dallas_df.count():,}")

Train: 9,474   Test: 2,436   Dallas: 5,913


#### Model and baseline helpers

The model is the same as before: GBT on `log1p(price)`, 100 trees, depth 5. Only the features change.
Empty numbers are filled with the median of the train data (`Imputer`).

In [0]:
def fit_model(df, numeric, categorical):
    filled = [c + "_filled" for c in numeric]
    stages = [Imputer(inputCols=numeric, outputCols=filled, strategy="median")]
    for c in categorical:
        stages.append(StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep"))
        stages.append(OneHotEncoder(inputCol=c + "_idx", outputCol=c + "_vec", handleInvalid="keep"))
    stages.append(VectorAssembler(inputCols=filled + [c + "_vec" for c in categorical], outputCol="features"))
    stages.append(GBTRegressor(featuresCol="features", labelCol="log_price", predictionCol="log_prediction",
                               maxIter=100, maxDepth=5, seed=RANDOM_SEED))
    return Pipeline(stages=stages).fit(df.withColumn("log_price", F.log1p("price")))


def predict(model, df):
    return model.transform(df).withColumn("predicted_price", F.expm1("log_prediction"))


def mae(df, column):
    return df.agg(F.avg(F.abs(F.col("price") - F.col(column)))).first()[0]


# Baseline: median train price for the same room_type and accommodates
baseline_lookup = (
    train_df.groupBy("room_type", "accommodates")
    .agg(F.expr("percentile_approx(price, 0.5)").alias("group_median"))
)
overall_median = train_df.approxQuantile("price", [0.5], 0.001)[0]


def add_baseline(df):
    return (
        df.join(baseline_lookup, on=["room_type", "accommodates"], how="left")
        .withColumn("baseline_price", F.coalesce("group_median", F.lit(overall_median)))
        .drop("group_median")
    )

#### Choose the features without Dallas

We train on one training city and test on the other one. This is like a "small new city" test, and it uses **only the train part** of Austin and Fort Worth.
The feature set with the lower average error wins.

In [0]:
check_rows = []
for name, (numeric, categorical) in FEATURE_SETS.items():
    for source, target in [("austin", "fort_worth"), ("fort_worth", "austin")]:
        model = fit_model(train_df.filter(F.col("city") == source), numeric, categorical)
        scored = predict(model, train_df.filter(F.col("city") == target))
        check_rows.append((name, f"{source} -> {target}", float(mae(scored, "predicted_price"))))

check = spark.createDataFrame(check_rows, "feature_set string, direction string, mae double")
display(check.groupBy("feature_set").pivot("direction").agg(F.round(F.first("mae"), 2)))

ranking = check.groupBy("feature_set").agg(F.avg("mae").alias("avg_mae")).orderBy("avg_mae").collect()
BEST = ranking[0]["feature_set"]
for r in ranking:
    print(f"{r['feature_set']:<28} average MAE in the other city: ${r['avg_mae']:.2f}")
print(f"\nChosen: {BEST}")

feature_set,austin -> fort_worth,fort_worth -> austin
v1: old features,67.8,119.75
v2: transferable features,68.94,108.64


v2: transferable features    average MAE in the other city: $88.79
v1: old features             average MAE in the other city: $93.77

Chosen: v2: transferable features


#### Train the final model and test it once

We train on the full train part of both cities with the chosen features. Then we test on the A/B test set and on Dallas.
For comparison we also score Dallas with the old saved model (`airbnb_v1`).

In [0]:
numeric, categorical = FEATURE_SETS[BEST]
final_model = fit_model(train_df, numeric, categorical)

test_scored = add_baseline(predict(final_model, test_df))
dallas_scored = add_baseline(predict(final_model, dallas_df))

# Old model on Dallas (the first run), with the same preparation as Texas_inference
v1_model = PipelineModel.load(V1_MODEL_DIR)
v1_fill = json.loads(dbutils.fs.head(V1_PREPROCESSING_JSON))["median_fill_values"]
dallas_v1 = v1_model.transform(dallas_df.fillna(v1_fill)).withColumn("v1_price", F.expm1("model_prediction"))
dallas_v1_mae = mae(dallas_v1, "v1_price")

# Old model on the A/B test set, from its metrics table
v1_metrics = {
    r["city"]: r["mae"]
    for r in spark.table(V1_METRICS_TABLE).filter(F.col("model_type") == "GBTRegressor").collect()
}

results = []
for city in TRAIN_CITIES + ["combined"]:
    part = test_scored if city == "combined" else test_scored.filter(F.col("city") == city)
    results.append((f"{city} (test)", float(mae(part, "baseline_price")), float(v1_metrics.get(city)), float(mae(part, "predicted_price"))))
results.append((f"{NEW_CITY} (all listings)", float(mae(dallas_scored, "baseline_price")), float(dallas_v1_mae), float(mae(dallas_scored, "predicted_price"))))

results_df = (
    spark.createDataFrame(results, "dataset string, baseline_mae double, v1_mae double, v2_mae double")
    .withColumn("v2_vs_baseline_pct", F.round(100 * (1 - F.col("v2_mae") / F.col("baseline_mae")), 1))
    .withColumn("v2_vs_v1_pct", F.round(100 * (1 - F.col("v2_mae") / F.col("v1_mae")), 1))
)
display(results_df.select("dataset", *[F.round(c, 2).alias(c) for c in ["baseline_mae", "v1_mae", "v2_mae"]],
                          "v2_vs_baseline_pct", "v2_vs_v1_pct"))

d = results[-1]
print(f"Dallas MAE: baseline ${d[1]:.2f} | first run (v1) ${d[2]:.2f} | after changes ({BEST.split(':')[0]}) ${d[3]:.2f}")

dataset,baseline_mae,v1_mae,v2_mae,v2_vs_baseline_pct,v2_vs_v1_pct
austin (test),111.42,79.75,77.01,30.9,3.4
fort_worth (test),92.25,60.49,60.44,34.5,0.1
combined (test),107.97,76.28,74.02,31.4,3.0
dallas (all listings),94.4,82.34,77.71,17.7,5.6


Dallas MAE: baseline $94.40 | first run (v1) $82.34 | after changes (v2) $77.71


#### Where the new model still makes big errors in Dallas

In [0]:
display(
    dallas_scored
    .withColumn("abs_error", F.abs(F.col("price") - F.col("predicted_price")))
    .select("id", "room_type", "accommodates", "dist_center_km", "has_pool",
            "price", F.round("predicted_price", 0).alias("predicted_price"), F.round("abs_error", 0).alias("abs_error"))
    .orderBy(F.desc("abs_error"))
    .limit(15)
)

id,room_type,accommodates,dist_center_km,has_pool,price,predicted_price,abs_error
1235446186014942010,Private room,1.0,5.075482921864548,0.0,1484.0,185.0,1299.0
721671330465940487,Entire home/apt,4.0,8.26445521219941,0.0,1500.0,288.0,1212.0
1688577612042934804,Entire home/apt,6.0,6.418188758144189,0.0,1500.0,403.0,1097.0
1678940527114701045,Entire home/apt,2.0,1.8754289259882762,1.0,1264.0,229.0,1035.0
1740301073531608186,Entire home/apt,5.0,4.50129774065785,0.0,1458.2,443.0,1015.0
1630645479093448240,Entire home/apt,2.0,13.652959873330428,0.0,1294.0,287.0,1007.0
1749789591705048632,Entire home/apt,14.0,5.229319768099252,0.0,1430.0,440.0,990.0
1685028026370143816,Entire home/apt,4.0,0.4099975204877358,0.0,1215.0,259.0,956.0
1204281581288123641,Private room,2.0,4.855837167870227,0.0,1232.0,299.0,933.0
851568353547676735,Entire home/apt,11.0,10.627138767472646,1.0,1166.18,237.0,929.0


#### Save the new model, predictions and metrics

New tables, so the old ones (`gold_price_predictions`, `gold_model_metrics`) stay as they are:

- `gold_price_predictions_v2`: one row per listing in all three cities (`split` = train / test / inference), the same columns as `gold_price_predictions`;
- `gold_model_metrics_v2`: the table above;
- the model in the volume: `price_model_airbnb_v2`.

In [0]:
final_model.write().overwrite().save(V2_MODEL_DIR)

train_scored = add_baseline(predict(final_model, train_df))

predictions_v2 = (
    train_scored.unionByName(test_scored).unionByName(dallas_scored)
    .select(
        "id", "city", "snapshot_date", "split", "room_type", "accommodates",
        F.col("price").alias("actual_price"),
        "predicted_price",
        "baseline_price",
        F.abs(F.col("price") - F.col("predicted_price")).alias("model_absolute_error"),
        F.abs(F.col("price") - F.col("baseline_price")).alias("baseline_absolute_error"),
        F.lit(MODEL_VERSION).alias("model_version"),
        F.current_timestamp().alias("prediction_timestamp"),
    )
)

predictions_v2.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(NEW_PREDICTIONS_TABLE)
results_df.withColumn("model_version", F.lit(MODEL_VERSION)).withColumn("features", F.lit(BEST)) \
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(NEW_METRICS_TABLE)

saved = spark.table(NEW_PREDICTIONS_TABLE)
assert saved.count() == data.count(), "Every listing must have one prediction"
assert saved.filter(F.col("predicted_price").isNull()).count() == 0, "Some predictions are empty"
display(saved.groupBy("city", "split").count().orderBy("city", "split"))
print(f"Saved: {V2_MODEL_DIR}, {NEW_PREDICTIONS_TABLE}, {NEW_METRICS_TABLE}")

city,split,count
austin,test,1997
austin,train,7762
dallas,inference,5913
fort_worth,test,439
fort_worth,train,1712


Saved: /Volumes/workspace/default/models/price_model_airbnb_v2, workspace.default.gold_price_predictions_v2, workspace.default.gold_model_metrics_v2


#### Conclusion

> **Fill in after the run** with the numbers from the table above:
> *"After the model changes, the MAE in Dallas went from \$__ (first run) to \$__. The baseline in Dallas is \$__.
> On the Austin + Fort Worth test set the MAE is \$__ (before: \$76.28)."*

The changes are **model changes**: the pipeline did not break; some features did not transfer to a new city. We chose the new features with Austin and Fort Worth only, so the Dallas number is still an honest first look at a city the model never saw.